In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import shutil
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from google.colab import drive
drive.mount('/content/drive')

# Pasta com as subpastas por ano (ex: .../2015/, .../2016/, ...)
PASTA_RAIZ = "/content/drive/MyDrive/PCD2/DadosClimaticos"

PASTA_LOCAL = "/content/dados_inmet"

def copiar_um_arquivo(origem, destino, tentativas=3):
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    for _ in range(tentativas):
        try:
            shutil.copy2(origem, destino)
            return True
        except Exception:
            time.sleep(2)
    return False

arquivos_origem = sorted(set(
    glob.glob(f"{PASTA_RAIZ}/**/*.CSV", recursive=True) +
    glob.glob(f"{PASTA_RAIZ}/**/*.csv", recursive=True)
))
print(f"{len(arquivos_origem)} arquivos encontrados no Drive.")

pares = [(o, o.replace(PASTA_RAIZ, PASTA_LOCAL)) for o in arquivos_origem]

falhas_copia = []
with ThreadPoolExecutor(max_workers=16) as executor:
    futuros = {executor.submit(copiar_um_arquivo, o, d): o for o, d in pares}
    for fut in tqdm(as_completed(futuros), total=len(futuros), desc="Copiando"):
        if not fut.result():
            falhas_copia.append(futuros[fut])

print(f"Cópia concluída. {len(falhas_copia)} falha(s) de cópia.")
for f in falhas_copia[:20]:
    print(f"  - {f}")

In [ ]:
# ============================================================
# CÉLULA 1 — Conectar ao Drive e definir caminhos
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

PASTA_LOCAL = "/content/dados_inmet"   # pasta local já copiada (ver script anterior)


# ============================================================
# CÉLULA 2 — Imports
# ============================================================
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Reaproveite aqui a função ler_arquivo_inmet() do script de união
# (copie a definição dela para esta célula, ou rode aquele script antes
# deste na mesma sessão do Colab para reutilizar a função já definida)


# ============================================================
# CÉLULA 3 — Diagnóstico leve por arquivo (roda em todos, mesmo 5000+)
# ============================================================
# Em vez de carregar todos os dados horários em memória, cada arquivo é
# lido, diagnosticado, e só um resumo pequeno (1 linha) é guardado.
# Isso permite fazer EDA em qualquer volume de arquivos sem estourar RAM.

# Faixas fisicamente plausíveis para o Brasil — ajuste se tiver estações
# em regiões com extremos conhecidos (ex: Sul tem mínimas mais baixas).
FAIXAS_PLAUSIVEIS = {
    'temperatura': (-10, 45),
    'precipitacao': (0, 150),   # mm em 1 hora
    'umidade': (0, 100),
    'vento': (0, 40),           # m/s
}


##################################
def normalizar_chave_meta(chave):
    """A partir de 2019 o INMET passou a exportar alguns cabeçalhos com
    caracteres corrompidos (ex: 'ESTAC?O' em vez de 'ESTAÇÃO'). Normaliza
    as variantes conhecidas para uma chave única."""
    chave = chave.strip().upper()
    correcoes = {
        'REGI?O': 'REGIAO', 'REGIÃO': 'REGIAO',
        'ESTAC?O': 'ESTACAO', 'ESTAÇÃO': 'ESTACAO',
        'DATA DE FUNDAC?O': 'DATA DE FUNDACAO', 'DATA DE FUNDAÇÃO': 'DATA DE FUNDACAO',
    }
    return correcoes.get(chave, chave)

COLUNAS_CONHECIDAS = {
  'PRECIPITAÇÃO TOTAL, HORÁRIO (MM)': 'precipitacao',
  'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'.upper(): 'temperatura',
  'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'.upper(): 'temp_max_hora',
  'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'.upper(): 'temp_min_hora',
  'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade',
  'RADIACAO GLOBAL (KJ/M²)': 'radiacao',
  'VENTO, VELOCIDADE HORARIA (M/S)': 'vento',
}

COLUNAS_ESSENCIAIS = {'data', 'hora', 'precipitacao', 'temperatura', 'temp_max_hora',
                       'temp_min_hora', 'umidade', 'radiacao', 'vento'}


def ler_arquivo_inmet(path):
    """Lê um único CSV do INMET (formato padrão de estação automática)
    e retorna um dataframe horário limpo, com metadados da estação.
    Compatível com o formato antigo (até ~2018) e os formatos 2019+, que
    mudaram nomes de coluna (inclusive maiúscula/minúscula) e corromperam
    alguns cabeçalhos de metadados. Levanta erro claro se alguma coluna
    essencial não for encontrada, em vez de falhar mais adiante."""
    meta = {}
    with open(path, encoding='latin1') as f:
        for _ in range(8):
            linha = f.readline().strip()
            chave, valor = linha.split(':;')
            meta[normalizar_chave_meta(chave)] = valor.strip()

    df = pd.read_csv(path, sep=';', encoding='latin1', skiprows=8, decimal=',')
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]  # remove coluna fantasma do ';' final

    mapa_renomeacao = {}
    for col in df.columns:
        col_upper = col.strip().upper()
        if col_upper.startswith('DATA'):
            mapa_renomeacao[col] = 'data'
        elif col_upper.startswith('HORA'):
            mapa_renomeacao[col] = 'hora'
        elif col_upper in COLUNAS_CONHECIDAS:
            mapa_renomeacao[col] = COLUNAS_CONHECIDAS[col_upper]

    df = df.rename(columns=mapa_renomeacao)

    faltando = COLUNAS_ESSENCIAIS - set(df.columns)
    if faltando:
        raise ValueError(f"Colunas essenciais não encontradas em {path}: {faltando}")

    df['data'] = pd.to_datetime(df['data'])

    colunas_numericas = df.select_dtypes(include=[np.number]).columns
    df[colunas_numericas] = df[colunas_numericas].replace(-9999, np.nan)

    # Radiação: -9999 à noite geralmente significa 0, não ausência real
    hora_num = df['hora'].astype(str).str.strip().str.slice(0, 2).astype(int)
    noite = (hora_num < 9) | (hora_num > 19)
    df.loc[df['radiacao'].isna() & noite, 'radiacao'] = 0
    df['radiacao'] = df['radiacao'].clip(lower=0)

    df['codigo_estacao'] = meta['CODIGO (WMO)']
    df['uf']             = meta['UF']
    df['estacao']        = meta['ESTACAO']
    df['latitude']       = float(meta['LATITUDE'].replace(',', '.'))
    df['longitude']      = float(meta['LONGITUDE'].replace(',', '.'))

    return df

def diagnosticar_arquivo(path):
    df = ler_arquivo_inmet(path)

    diag = {
        'arquivo': path.split('/')[-1],
        'codigo_estacao': df['codigo_estacao'].iloc[0],
        'estacao': df['estacao'].iloc[0],
        'uf': df['uf'].iloc[0],
        'ano': df['data'].dt.year.mode()[0],
        'n_horas': len(df),
        'periodo_min': df['data'].min(),
        'periodo_max': df['data'].max(),
    }

    for var in ['temperatura', 'precipitacao', 'umidade', 'vento', 'radiacao']:
        diag[f'{var}_pct_faltante'] = round(df[var].isna().mean() * 100, 1)

    for var, (lo, hi) in FAIXAS_PLAUSIVEIS.items():
        fora = ((df[var] < lo) | (df[var] > hi)).sum()
        diag[f'{var}_fora_faixa'] = int(fora)

    # Maior sequência de horas consecutivas faltando (gap "burro" de
    # sensor, diferente de falha aleatória isolada)
    faltando = df['temperatura'].isna()
    grupos = (~faltando).cumsum()
    maior_gap = faltando.groupby(grupos).sum().max() if faltando.any() else 0
    diag['maior_gap_horas_temp'] = int(maior_gap)

    return diag


arquivos = sorted(
    glob.glob(f"{PASTA_LOCAL}/**/*.CSV", recursive=True) +
    glob.glob(f"{PASTA_LOCAL}/**/*.csv", recursive=True)
)

diagnosticos, erros_diag = [], []
for path in tqdm(arquivos, desc="Diagnosticando"):
    try:
        diagnosticos.append(diagnosticar_arquivo(path))
    except Exception as e:
        erros_diag.append((path, str(e)))

diag_df = pd.DataFrame(diagnosticos)
print(f"{len(diag_df)} arquivos diagnosticados, {len(erros_diag)} com erro.")


# ============================================================
# CÉLULA 4 — Ranking dos piores casos
# ============================================================
print("\nTop 15 arquivos com mais dados faltando (temperatura):")
print(diag_df.sort_values('temperatura_pct_faltante', ascending=False)
      [['arquivo', 'codigo_estacao', 'ano', 'temperatura_pct_faltante']].head(15))

print("\nTop 15 arquivos com maior gap contínuo (horas seguidas sem dado):")
print(diag_df.sort_values('maior_gap_horas_temp', ascending=False)
      [['arquivo', 'codigo_estacao', 'ano', 'maior_gap_horas_temp']].head(15))

colunas_fora_faixa = [c for c in diag_df.columns if c.endswith('_fora_faixa')]
diag_df['total_fora_faixa'] = diag_df[colunas_fora_faixa].sum(axis=1)
print("\nTop 15 arquivos com mais valores fisicamente implausíveis:")
print(diag_df.sort_values('total_fora_faixa', ascending=False)
      [['arquivo', 'codigo_estacao', 'ano'] + colunas_fora_faixa].head(15))

# Completude por estação ao longo dos anos (útil para decidir se uma
# estação tem histórico bom o suficiente para entrar na agregação por UF)
print("\nCompletude média de temperatura por estação, todos os anos:")
print(diag_df.groupby('codigo_estacao')['temperatura_pct_faltante'].mean().sort_values(ascending=False))


# ============================================================
# CÉLULA 5 — Visualização geral (a partir dos diagnósticos, leve)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

diag_df['temperatura_pct_faltante'].hist(bins=30, ax=axes[0, 0])
axes[0, 0].set_title('Distribuição de % faltante de temperatura por arquivo')

diag_df['maior_gap_horas_temp'].hist(bins=30, ax=axes[0, 1])
axes[0, 1].set_title('Distribuição do maior gap contínuo (horas)')

diag_df.groupby('ano')['temperatura_pct_faltante'].mean().plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('% faltante médio de temperatura por ano')

diag_df.groupby('uf')['n_horas'].count().sort_values(ascending=False).plot(kind='bar', ax=axes[1, 1])
axes[1, 1].set_title('Nº de arquivos (estação-ano) por UF')

plt.tight_layout()
plt.show()


# ============================================================
# CÉLULA 6 — Inspeção detalhada de um caso flagrado (só esse, não todos)
# ============================================================
# Depois de identificar um arquivo suspeito na Célula 4, carregue só ele
# (não os 5000) para olhar a série temporal horária de verdade.

def plotar_serie_horaria(path, variavel='temperatura'):
    df = ler_arquivo_inmet(path)
    df = df.sort_values('data')
    timestamp = pd.to_datetime(df['data'].astype(str) + ' ' + df['hora'].astype(str).str.slice(0, 2))

    plt.figure(figsize=(14, 4))
    plt.plot(timestamp, df[variavel], linewidth=0.5)
    plt.title(f"{variavel} — {df['estacao'].iloc[0]} ({df['codigo_estacao'].iloc[0]})")
    plt.xlabel('Data/hora')
    plt.ylabel(variavel)
    plt.tight_layout()
    plt.show()

# Exemplo de uso — troque pelo path de um arquivo flagrado na Célula 4:
# plotar_serie_horaria(arquivos[0], variavel='temperatura')
# plotar_serie_horaria(arquivos[0], variavel='precipitacao')

In [ ]:
plotar_serie_horaria(arquivos[0], variavel='temperatura')
plotar_serie_horaria(arquivos[0], variavel='precipitacao')

Análise do arquivo agregado

In [ ]:
# ============================================================
# CÉLULA 1 — Conectar ao Drive e carregar o diário unido
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

caminho_diario = "/content/drive/MyDrive/PCD2/DadosClimaticos/processado/clima_diario.parquet"
diario = pd.read_parquet(caminho_diario)
diario['data'] = pd.to_datetime(diario['data'])
diario['ano'] = diario['data'].dt.year
diario['mes'] = diario['data'].dt.month

print("Shape:", diario.shape)
print("Período:", diario['data'].min().date(), "a", diario['data'].max().date())
print("Estações:", diario['codigo_estacao'].nunique(), "| UFs:", diario['uf'].nunique())
diario.head()


# ============================================================
# CÉLULA 2 — Visão geral: completude e cobertura temporal
# ============================================================
print("Nulos por coluna (%):")
print((diario.isna().mean() * 100).round(1).sort_values(ascending=False))

# Quantos dias de dado cada estação tem, por ano — mostra buracos de
# cobertura (ex: estação que só tem 3 meses de dado num ano específico)
cobertura = diario.groupby(['codigo_estacao', 'ano']).size().unstack(fill_value=0)
print("\nCobertura (dias com dado válido) por estação x ano:")
print(cobertura)

estacoes_uf = df_diario.groupby('uf')['codigo_estacao'].nunique().sort_values(ascending=False)
print(estacoes_uf)


plt.figure(figsize=(12, max(3, 0.35 * len(cobertura))))
plt.imshow(cobertura, aspect='auto', cmap='YlGnBu')
plt.colorbar(label='dias com dado válido')
plt.yticks(range(len(cobertura.index)), cobertura.index)
plt.xticks(range(len(cobertura.columns)), cobertura.columns, rotation=45)
plt.title('Cobertura diária por estação x ano')
plt.tight_layout()
plt.show()


# ============================================================
# CÉLULA 3 — Distribuições das variáveis principais
# ============================================================
variaveis = ['prec_total', 'temp_media', 'temp_max', 'temp_min',
             'umid_media', 'amplitude_termica', 'vento_medio']
variaveis = [v for v in variaveis if v in diario.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, var in enumerate(variaveis):
    diario[var].hist(bins=40, ax=axes[i])
    axes[i].set_title(var)
for j in range(len(variaveis), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

print("\nEstatísticas descritivas:")
print(diario[variaveis].describe().round(2))


# ============================================================
# CÉLULA 4 — Séries temporais (uma estação por vez, ou média nacional)
# ============================================================
# Ajuste ESTACAO_FOCO para o código de uma estação específica, ou deixe
# None para ver a média diária de todas as estações combinadas.
ESTACAO_FOCO = None  # ex: 'A001'

if ESTACAO_FOCO:
    serie = diario[diario['codigo_estacao'] == ESTACAO_FOCO].sort_values('data')
    titulo_sufixo = f"— estação {ESTACAO_FOCO}"
else:
    serie = diario.groupby('data')[['prec_total', 'temp_media']].mean().reset_index()
    titulo_sufixo = "— média de todas as estações"

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Insere NaN em datas com mais de 3 dias de gap, para o matplotlib não
# desenhar uma linha reta "interpolada" sobre períodos sem dado nenhum
# (ex: anos inteiros faltando entre dois arquivos disponíveis).
serie_completa = serie.set_index('data').asfreq('D')
gap = serie_completa.index.to_series().diff()
axes[0].plot(serie_completa.index, serie_completa['temp_media'], linewidth=0.6, color='tomato')
axes[0].set_title(f"Temperatura média diária {titulo_sufixo}")
axes[0].xaxis.set_major_locator(mdates.YearLocator())

axes[1].bar(serie['data'], serie['prec_total'], width=1, color='steelblue')
axes[1].set_title(f"Precipitação total diária {titulo_sufixo}")

plt.tight_layout()
plt.show()


# ============================================================
# CÉLULA 5 — Sazonalidade: boxplot por mês (todas as estações e anos juntos)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
diario.boxplot(column='temp_media', by='mes', ax=axes[0])
axes[0].set_title('Temperatura média por mês')
axes[0].set_xlabel('Mês')
plt.suptitle('')

diario.boxplot(column='prec_total', by='mes', ax=axes[1])
axes[1].set_title('Precipitação total por mês')
axes[1].set_xlabel('Mês')
plt.suptitle('')

plt.tight_layout()
plt.show()


# ============================================================
# CÉLULA 6 — Comparação entre estações/UFs
# ============================================================
resumo_estacao = diario.groupby(['codigo_estacao', 'uf']).agg(
    temp_media=('temp_media', 'mean'),
    prec_media_diaria=('prec_total', 'mean'),
    n_dias=('data', 'count'),
).reset_index().sort_values('temp_media')

print("Resumo por estação:")
print(resumo_estacao)

if diario['uf'].nunique() > 1:
    plt.figure(figsize=(10, 5))
    diario.boxplot(column='temp_media', by='uf')
    plt.title('Temperatura média diária por UF')
    plt.suptitle('')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# ============================================================
# CÉLULA 7 — Correlação entre variáveis climáticas
# ============================================================
corr = diario[variaveis].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(variaveis)))
ax.set_xticklabels(variaveis, rotation=45, ha='right')
ax.set_yticks(range(len(variaveis)))
ax.set_yticklabels(variaveis)
for i in range(len(variaveis)):
    for j in range(len(variaveis)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha='center', va='center', fontsize=8)
plt.colorbar(im, label='correlação')
plt.title('Correlação entre variáveis climáticas diárias')
plt.tight_layout()
plt.show()


# ============================================================
# CÉLULA 8 — Outliers: dias extremos por variável
# ============================================================
for var in ['prec_total', 'temp_max', 'temp_min']:
    print(f"\nTop 5 valores mais altos de {var}:")
    print(diario.nlargest(5, var)[['data', 'codigo_estacao', 'uf', var]])

In [ ]:
#estações que aparecem ao menos em 300 dias
(
    df_diario
    .groupby(['ano', 'codigo_estacao'])
    .size()
    .groupby('ano')
    .apply(lambda x: (x >= 300).sum())
)

In [ ]:
estacoes_por_ano = df_diario.groupby('ano')['codigo_estacao'].nunique()

print(estacoes_por_ano)

In [ ]:
df_diario.groupby('uf')['codigo_estacao'].nunique().sort_values(ascending=False)